# Strategy Pipeline Runner

Thin notebook on top of the `aapl_pipeline` package. The two original
setups share the same backtest layout (linspace subsets after
`subset_start_date`, historical Monte Carlo over prior history,
future Monte Carlo over the entire df_final). The only difference
between them is the feature calculator and the parameters that drive
it. Every parameter is a runtime input you can override per run.

Two factory helpers cover the common cases:
1. `make_tue_thu_pipeline` uses the three normalized opens.
2. `make_support_resistance_pipeline` uses the nine envelope features.

In [ ]:
import sys
from pathlib import Path
import numpy as np

# When running from notebooks/ we need the project root on sys.path so
# the package import works without pip install. If the package is
# already installed via pyproject.toml, this block is a harmless no op.
_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from aapl_pipeline import (
    PipelineConfig,
    StrategyPipeline,
    SupportResistanceFeatures,
    TueThuNormalizedFeatures,
    DecisionTreeModeller,
    RandomForestModeller,
    make_tue_thu_pipeline,
    make_support_resistance_pipeline,
)

## 1. Tuesday Thursday normalized features

Same backtest layout as the SR pipeline. Pass the params as a flat
dict and let `from_params_dict` convert them. Both `n_subsets` and
`num_subsets` keys are accepted for the subset count.

In [ ]:
params_tue_thu = {
    "VALID_WEEKS": 52,
    "depth_grid": [2, 3, 4, 5, 6],
    "leaf_grid": [2, 3, 4, 5, 6],
    "thresholds_tested": np.linspace(0.01, 0.99, 99),
    "FIXED": {
        "criterion": "entropy",
        "min_samples_split": 6,
        "class_weight": "balanced",
        "random_state": 42,
    },
    "alpha_p": 1.0,
    "alpha_c": 0.01,
    "p_min": 0.55,
    "c_min": 0.10,
    "n_trajectories": 100000,
    "n_weeks": 100,
    "initial_bank": 100.0,
    "upper_thresh": 200.0,
    "lower_thresh": 60.0,
    "rng_seed": 42,
    "uniformity_binsize": 104,
    "cutoff_date": "1990-01-01",
    "subset_start_date": "2000-01-01",
    "num_subsets": 5,
}

config_tue_thu = PipelineConfig.from_params_dict(params_tue_thu)
pipe_tt = make_tue_thu_pipeline(config_tue_thu)
report_tt = pipe_tt.run()

## 2. Support resistance features

Identical backtest layout. The extra parameters at the bottom
(ENVELOPE_A through SR_WINDOW_VALS) are read by
`SupportResistanceFeatures` and do not affect anything outside the
feature calculator.

In [ ]:
params_sr = {
    "VALID_WEEKS": 52,
    "depth_grid": [2, 3, 4, 6],
    "leaf_grid": [2, 3, 4, 6],
    "thresholds_tested": np.linspace(0.01, 0.99, 99),
    "FIXED": {
        "criterion": "entropy",
        "min_samples_split": 6,
        "class_weight": "balanced",
        "random_state": 42,
    },
    "alpha_p": 1.0,
    "alpha_c": 0.01,
    "p_min": 0.55,
    "c_min": 0.10,
    "n_trajectories": 100000,
    "n_weeks": 100,
    "initial_bank": 100.0,
    "upper_thresh": 200.0,
    "lower_thresh": 60.0,
    "rng_seed": 42,
    "uniformity_binsize": 104,
    "cutoff_date": "1990-01-01",
    "subset_start_date": "2000-01-01",
    "num_subsets": 5,
    "ENVELOPE_A": 1.0,
    "ENVELOPE_B": 100.0,
    "SR_MAX_ITER": 60,
    "SR_TOL": 1e-9,
    "SR_SMOOTH_VALS": [2, 4, 12],
    "SR_WINDOW_VALS": [4, 26, 52],
}

config_sr = PipelineConfig.from_params_dict(params_sr)
pipe_sr = make_support_resistance_pipeline(config_sr)
report_sr = pipe_sr.run()

## 3. Both feature families together

Concatenated features go into the same model. The pipeline does not
need to know how many calculators were registered.

In [ ]:
config_both = PipelineConfig(
    cutoff_date="1990-01-01",
    subset_start_date="2000-01-01",
    num_subsets=5,
)
pipe_both = StrategyPipeline(
    config=config_both,
    feature_calculators=[
        TueThuNormalizedFeatures(),
        SupportResistanceFeatures(),
    ],
)
# pipe_both.run()  # commented to keep this notebook fast on a fresh cold run

## Inspecting artifacts

After a run the pipeline keeps the per week predictions, the Monte
Carlo arrays and the baseline ratios so you can dig past the report.

In [ ]:
df_final = pipe_sr.last_df_final
df_final.head()

In [ ]:
future_sims = pipe_sr.last_future_sims
print('future bank quartiles:', np.quantile(future_sims, [0.25, 0.5, 0.75]))

## Adding a new component

1. New feature family: subclass `FeatureCalculator`, fill in
   `feature_names` and `compute(daily_df, weekly_df)`, drop into
   `feature_calculators=[...]`.
2. New model: subclass `Modeller`, fill in `make_estimator(**hp)`
   and `param_grid()`, pass via `modeller=...`.
3. New metric: subclass `PerformanceMetric`, set `group` and `name`,
   register on a `MetricRegistry` and pass via `registry=...`.

None of these require touching `pipeline.py` or any existing class.